# Project 03 — Credit Risk: PD / LGD / EAD & Expected Loss
## Notebook 03 — Loss Given Default (LGD) & Exposure at Default (EAD)

This notebook extends the Probability of Default (PD) framework developed in Notebook 02 by examining the two remaining components of expected credit loss: Loss Given Default (LGD) and Exposure at Default (EAD).

LGD measures the proportion of exposure lost after accounting for recoveries, while EAD represents the outstanding exposure when a borrower defaults.

The notebook reuses the prepared development and validation datasets from Notebook 01. The analysis focuses on assessing available recovery information, establishing a transparent LGD framework, and defining an appropriate exposure measure.

Where the available data does not support direct estimation, simplified assumptions and proxies will be used and explicitly documented.

The resulting measures will provide the inputs required to combine PD, LGD, and EAD into an expected credit loss framework.

## 1. LGD and EAD Framework

Expected credit loss combines three components:

$$
EL_i = PD_i \times LGD_i \times EAD_i
$$

Where:

- **PD (Probability of Default):** Probability that borrower \(i\) defaults over the defined outcome horizon.
- **LGD (Loss Given Default):** Proportion of exposure lost after accounting for recoveries.
- **EAD (Exposure at Default):** Outstanding exposure when the borrower defaults.
- **EL (Expected Loss):** Expected monetary loss associated with the exposure.

PD addresses the likelihood of default, LGD addresses the severity of loss, and EAD measures the amount exposed to that loss.

The objective of this notebook is to establish interpretable LGD and EAD measures without introducing unnecessary modeling complexity.

### 1.1 Data Availability and Initial Checks

In [2]:

import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("../data/processed")

development_path = data_dir / "credit_risk_development.csv"
validation_path = data_dir / "credit_risk_validation.csv"

train_df = pd.read_csv(development_path)
valid_df = pd.read_csv(validation_path)

print(f"Development sample: {len(train_df):,} loans")
print(f"Validation sample: {len(valid_df):,} loans")

print("\nDataset columns:")
print(train_df.columns.tolist())

relevant_columns = [
    col for col in [
        "default_flag",
        "loan_status",
        "loan_amnt",
        "funded_amnt",
        "recoveries",
        "grade",
        "issue_d",
    ]
    if col in train_df.columns
]

print("\nRelevant field summary:")
display(train_df[relevant_columns].head())

print("\nMissing values:")
display(train_df[relevant_columns].isna().sum().to_frame("Missing count"))

print("\nRecovery and exposure statistics:")
display(
    train_df[
        [c for c in ["loan_amnt", "funded_amnt", "recoveries"]
         if c in train_df.columns]
    ].describe().T
)


Development sample: 1,495,018 loans
Validation sample: 368,062 loans

Dataset columns:
['id', 'loan_amnt', 'funded_amnt', 'int_rate', 'grade', 'emp_length', 'home_ownership', 'annual_inc', 'issue_d', 'loan_status', 'purpose', 'dti', 'fico_range_low', 'fico_range_high', 'recoveries', 'default_flag', 'fico_avg', 'loan_to_income']

Relevant field summary:


,default_flag,loan_status,loan_amnt,funded_amnt,recoveries,grade,issue_d
0,0,Does not meet the credit policy. Status:Fully ...,1400.0,1400.0,0.0,C,2007-06-01
1,0,Fully Paid,7500.0,7500.0,0.0,E,2007-06-01
2,0,Does not meet the credit policy. Status:Fully ...,6000.0,6000.0,0.0,C,2007-06-01
3,0,Does not meet the credit policy. Status:Fully ...,4400.0,4400.0,0.0,B,2007-06-01
4,0,Does not meet the credit policy. Status:Fully ...,1200.0,1200.0,0.0,B,2007-06-01



Missing values:


,Missing count
default_flag,0
loan_status,0
loan_amnt,0
funded_amnt,0
recoveries,0
grade,0
issue_d,0



Recovery and exposure statistics:


,count,mean,std,min,25%,50%,75%,max
loan_amnt,1495018.0,14452.682978,8702.275809,500.0,8000.0,12000.0,20000.0,40000.00
funded_amnt,1495018.0,14444.724997,8698.804408,500.0,8000.0,12000.0,20000.0,40000.00
recoveries,1495018.0,268.709938,1061.000797,0.0,0.0,0.0,0.0,39859.55


### 1.2 Assess Recovery Information

The dataset contains a `recoveries` field representing amounts recovered after charge-off. However, recovery amounts alone are insufficient to establish realized LGD because the exposure outstanding at default is not directly available.

The recovery distribution will first be examined across defaulted and non-defaulted loans. This will help establish whether the field provides useful information for the LGD analysis.

Any ratio calculated using original funded amounts will be treated as an exploratory proxy rather than a validated estimate of realized LGD.

In [3]:

# Examine default outcomes and recovery amounts.

print("Default flag distribution:")
display(
    train_df["default_flag"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("default_flag")
    .to_frame("loan_count")
    .assign(
        percentage=lambda x: (
            x["loan_count"] / len(train_df) * 100
        ).round(2)
    )
)

print("\nLoan status distribution:")
display(
    train_df["loan_status"]
    .value_counts(dropna=False)
    .rename_axis("loan_status")
    .to_frame("loan_count")
    .head(15)
)

print("\nRecovery statistics by default flag:")
display(
    train_df.groupby("default_flag")["recoveries"]
    .agg(
        loan_count="count",
        mean_recoveries="mean",
        median_recoveries="median",
        total_recoveries="sum",
        positive_recovery_rate=lambda x: (x > 0).mean() * 100
    )
    .round(2)
)


Default flag distribution:


,loan_count,percentage
default_flag,,
0,1214412,81.23
1,280606,18.77



Loan status distribution:


,loan_count
loan_status,
Fully Paid,1212424
Charged Off,279845
Does not meet the credit policy. Status:Fully Paid,1988
Does not meet the credit policy. Status:Charged Off,761



Recovery statistics by default flag:


,loan_count,mean_recoveries,median_recoveries,total_recoveries,positive_recovery_rate
default_flag,,,,,
0,1214412,0.00,0.00,0.000000e+00,0.00
1,280606,1431.64,791.61,4.017262e+08,77.92


In [4]:

defaulted_loans = train_df.loc[
    train_df["default_flag"] == 1
].copy()

print(f"Defaulted loans: {len(defaulted_loans):,}")
print(
    "Defaulted loans with positive recoveries:",
    f"{(defaulted_loans['recoveries'] > 0).sum():,}"
)

print("\nRecovery distribution among defaulted loans:")
display(
    defaulted_loans["recoveries"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .to_frame("recoveries")
)

print("\nExposure and recovery summary:")
display(
    defaulted_loans[
        ["loan_amnt", "funded_amnt", "recoveries"]
    ].describe().T.round(2)
)


Defaulted loans: 280,606
Defaulted loans with positive recoveries: 218,655

Recovery distribution among defaulted loans:


,recoveries
count,280606.000000
mean,1431.637935
std,2081.529827
min,0.000000
25%,16.522500
50%,791.610000
75%,1911.930000
90%,3520.000000
95%,4991.940000
99%,10173.731500



Exposure and recovery summary:


,count,mean,std,min,25%,50%,75%,max
loan_amnt,280606.0,15560.95,8733.09,500.0,9000.00,14400.00,20400.00,40000.00
funded_amnt,280606.0,15552.55,8729.79,500.0,9000.00,14400.00,20400.00,40000.00
recoveries,280606.0,1431.64,2081.53,0.0,16.52,791.61,1911.93,39859.55


**Interpretation**

      Recoveries are concentrated in the defaulted-loan population: none of the non-defaulted loans has a positive recovery amount, while 77.92% of defaulted loans have positive recoveries. The median recovery among defaulted loans is $791.61.

      This confirms that `recoveries` provides useful information about post-charge-off collections. However, because the outstanding balance at default is unavailable, these figures alone cannot establish realized LGD.

### 1.3 Exploratory Recovery-Based LGD Proxy

The dataset provides recovery amounts for defaulted loans but does not contain outstanding balances at the point of default.

To assess the available recovery information, an exploratory recovery ratio will be calculated using the original funded amount as the exposure proxy:

$$
\text{Recovery Ratio}_{\text{proxy}} = \frac{\sum_i \text{Recoveries}_i}{\sum_i \text{Funded Amount}_i}
$$

The corresponding proxy LGD is:

$$
\text{LGD}_{\text{proxy}} = 1 - \text{Recovery Ratio}_{\text{proxy}}
$$

This calculation uses aggregate amounts rather than averaging individual loan-level ratios. It is an exploratory proxy, not a validated estimate of realized LGD, because original funded amounts may differ from outstanding exposure at default.

The result will be used to inform the LGD methodology, while the final framework will clearly distinguish empirical observations from assumption-based LGD values.

In [5]:

# Calculate an exploratory recovery-based LGD proxy
# for loans classified as defaults.

defaulted_loans = train_df.loc[
    train_df["default_flag"] == 1
].copy()

total_funded = defaulted_loans["funded_amnt"].sum()
total_recoveries = defaulted_loans["recoveries"].sum()

recovery_ratio_proxy = total_recoveries / total_funded
lgd_proxy = 1 - recovery_ratio_proxy

recovery_summary = pd.DataFrame({
    "Metric": [
        "Defaulted loans",
        "Total funded amount",
        "Total recoveries",
        "Recovery ratio proxy",
        "LGD proxy",
    ],
    "Value": [
        len(defaulted_loans),
        total_funded,
        total_recoveries,
        recovery_ratio_proxy,
        lgd_proxy,
    ],
})

display(recovery_summary)

print(
    f"Recovery ratio proxy: {recovery_ratio_proxy:.2%}"
)
print(f"Exploratory LGD proxy: {lgd_proxy:.2%}")


,Metric,Value
0,Defaulted loans,2.806060e+05
1,Total funded amount,4.364137e+09
2,Total recoveries,4.017262e+08
3,Recovery ratio proxy,9.205168e-02
4,LGD proxy,9.079483e-01


Recovery ratio proxy: 9.21%
Exploratory LGD proxy: 90.79%


### 1.4 LGD Methodology Decision

The exploratory analysis produces a recovery ratio proxy of 9.21% and an implied LGD proxy of 90.79%, calculated using aggregate recoveries and funded amounts for defaulted loans.

However, funded amounts represent original loan exposure rather than the outstanding balance at default. Consequently, the resulting LGD proxy should not be interpreted as a validated estimate of realized loss severity.

Given this limitation, the notebook adopts a segmented, assumption-based LGD framework for the expected-loss analysis.

The LGD assumptions are defined using the lender-assigned credit grade:

| Risk Segment | Credit Grades | Assumed LGD |
|---|---|---:|
| Low Risk | A–B | 40% |
| Medium Risk | C–D | 50% |
| High Risk | E–F | 60% |
| Very High Risk | G | 70% |

These values are illustrative assumptions, not statistically estimated LGD parameters. The segmentation reflects an assumed relationship between weaker credit grades and greater loss severity.

The empirical recovery-based proxy is retained as a diagnostic reference and is not used as the assigned LGD value.

In [6]:

# Define transparent, assumption-based LGD values by credit grade.

lgd_by_grade = {
    "A": 0.40,
    "B": 0.40,
    "C": 0.50,
    "D": 0.50,
    "E": 0.60,
    "F": 0.60,
    "G": 0.70,
}

train_df["lgd_assumption"] = train_df["grade"].map(lgd_by_grade)
valid_df["lgd_assumption"] = valid_df["grade"].map(lgd_by_grade)

# Check whether all grades were mapped successfully.
print("Unmapped development grades:",
      train_df["lgd_assumption"].isna().sum())

print("Unmapped validation grades:",
      valid_df["lgd_assumption"].isna().sum())

lgd_summary = (
    train_df.groupby("grade", observed=True)
    .agg(
        loan_count=("grade", "size"),
        assumed_lgd=("lgd_assumption", "first"),
    )
    .reset_index()
)

display(lgd_summary)


Unmapped development grades: 0
Unmapped validation grades: 0


,grade,loan_count,assumed_lgd
0,A,255941,0.4
1,B,445033,0.4
2,C,432460,0.5
3,D,215979,0.5
4,E,101745,0.6
5,F,34302,0.6
6,G,9558,0.7


These values are illustrative placeholders for demonstrating the expected-loss calculation. The assumed increase in LGD across weaker credit grades has not been empirically established in this dataset and should be validated or subjected to sensitivity analysis before the framework is used for decision-making.

## 2. Exposure at Default (EAD)

Exposure at Default (EAD) represents the amount of credit exposure outstanding when a borrower defaults.

For an installment-loan portfolio, EAD is conceptually defined as:

$$
EAD_i = Outstanding\ Balance_i
$$

The available dataset contains `loan_amnt` and `funded_amnt`, but does not provide the outstanding balance at default.

Therefore, this notebook uses `funded_amnt` as a simplified exposure proxy:

$$
EAD_{i,proxy} = Funded\ Amount_i
$$

This proxy represents the original amount funded by the lender, not the actual outstanding balance at default. It does not account for principal repayments before default.

The measure is used for illustrative expected-loss calculations, with this limitation explicitly acknowledged.

In [7]:

# Use the original funded amount as a simplified EAD proxy.

train_df["ead_proxy"] = train_df["funded_amnt"]
valid_df["ead_proxy"] = valid_df["funded_amnt"]

ead_summary = pd.DataFrame({
    "Metric": [
        "Development loans",
        "Validation loans",
        "Mean funded amount",
        "Median funded amount",
        "Minimum funded amount",
        "Maximum funded amount",
    ],
    "Value": [
        len(train_df),
        len(valid_df),
        train_df["ead_proxy"].mean(),
        train_df["ead_proxy"].median(),
        train_df["ead_proxy"].min(),
        train_df["ead_proxy"].max(),
    ],
})

display(ead_summary.round(2))

print(
    "Missing EAD proxies in development:",
    train_df["ead_proxy"].isna().sum()
)
print(
    "Missing EAD proxies in validation:",
    valid_df["ead_proxy"].isna().sum()
)


,Metric,Value
0,Development loans,1495018.00
1,Validation loans,368062.00
2,Mean funded amount,14444.72
3,Median funded amount,12000.00
4,Minimum funded amount,500.00
5,Maximum funded amount,40000.00


Missing EAD proxies in development: 0
Missing EAD proxies in validation: 0


### 2.1 EAD Proxy Interpretation

The funded amounts range from $500 to $40,000, with a median of $12,000. The field has no missing values in either sample, so it provides a consistent exposure proxy. However, these are amounts originally funded rather than balances outstanding at default.

## 3. Expected Credit Loss

Expected Loss (EL) combines Probability of Default (PD), Loss Given Default (LGD), and Exposure at Default (EAD).

$$
EL_i = PD_i \times LGD_i \times EAD_i
$$

For this analysis:

- PD is obtained from the PD model developed in Notebook 02.
- LGD is assigned using the assumption-based credit-grade framework.
- EAD is represented by the original funded amount proxy.

The resulting expected loss is an illustrative estimate based on the selected PD horizon and the LGD and EAD assumptions. It is not a regulatory expected-credit-loss estimate.

### 3.1 Import PD Predictions from Notebook 02

In [10]:

# Load PD predictions exported from Notebook 02.

train_pd = pd.read_csv(
    data_dir / "credit_risk_development_pd.csv"
)

valid_pd = pd.read_csv(
    data_dir / "credit_risk_validation_pd.csv"
)

print(f"Development PD predictions: {len(train_pd):,}")
print(f"Validation PD predictions: {len(valid_pd):,}")

# Validate identifiers before merging.
assert train_pd["id"].is_unique, "Duplicate development IDs found."
assert valid_pd["id"].is_unique, "Duplicate validation IDs found."

# Merge predictions into the existing datasets.
train_df = train_df.merge(
    train_pd,
    on="id",
    how="left",
    validate="one_to_one",
)

valid_df = valid_df.merge(
    valid_pd,
    on="id",
    how="left",
    validate="one_to_one",
)

# Verify prediction coverage.
print(
    "\nMissing development predictions:",
    train_df["predicted_pd"].isna().sum(),
)

print(
    "Missing validation predictions:",
    valid_df["predicted_pd"].isna().sum(),
)

assert train_df["predicted_pd"].notna().all()
assert valid_df["predicted_pd"].notna().all()

display(
    train_df[
        ["id", "predicted_pd", "lgd_assumption", "ead_proxy"]
    ].head()
)


Development PD predictions: 1,495,018
Validation PD predictions: 368,062

Missing development predictions: 0
Missing validation predictions: 0


,id,predicted_pd,lgd_assumption,ead_proxy
0,85781,0.158369,0.5,1400.0
1,87023,0.255729,0.6,7500.0
2,88637,0.176585,0.5,6000.0
3,88046,0.082347,0.4,4400.0
4,85961,0.088660,0.4,1200.0


### 3.2 Expected Credit Loss

Expected Credit Loss (EL) combines the Probability of Default (PD), Loss Given Default (LGD), and Exposure at Default (EAD).

$$
EL_i = PD_i \times LGD_i \times EAD_i
$$

For each loan, expected loss is calculated using the predicted PD from Notebook 02, the assumption-based LGD assigned by credit grade, and the funded amount as an EAD proxy.

The resulting measure represents an illustrative expected loss under the selected assumptions. Since the EAD proxy does not reflect outstanding balances at default and LGD is assumption-based, the results should not be interpreted as validated estimates of realized credit losses.

In [11]:

# Calculate expected loss for each loan.

train_df["expected_loss"] = (
    train_df["predicted_pd"]
    * train_df["lgd_assumption"]
    * train_df["ead_proxy"]
)

valid_df["expected_loss"] = (
    valid_df["predicted_pd"]
    * valid_df["lgd_assumption"]
    * valid_df["ead_proxy"]
)

# Summarize portfolio-level expected loss.
def summarize_expected_loss(df, sample_name):
    return {
        "Sample": sample_name,
        "Loan Count": len(df),
        "Total Exposure Proxy": df["ead_proxy"].sum(),
        "Total Expected Loss": df["expected_loss"].sum(),
        "Average Expected Loss per Loan": df["expected_loss"].mean(),
        "Expected Loss / Exposure": (
            df["expected_loss"].sum()
            / df["ead_proxy"].sum()
        ),
    }

el_summary = pd.DataFrame([
    summarize_expected_loss(train_df, "Development"),
    summarize_expected_loss(valid_df, "Validation"),
])

display(
    el_summary.style.format({
        "Loan Count": "{:,.0f}",
        "Total Exposure Proxy": "${:,.2f}",
        "Total Expected Loss": "${:,.2f}",
        "Average Expected Loss per Loan": "${:,.2f}",
        "Expected Loss / Exposure": "{:.2%}",
    })
)


,Sample,Loan Count,Total Exposure Proxy,Total Expected Loss,Average Expected Loss per Loan,Expected Loss / Exposure
0,Development,"1,495,018","$21,595,123,875.00","$2,200,968,576.86","$1,472.20",10.19%
1,Validation,"368,062","$5,556,844,900.00","$535,593,371.63","$1,455.17",9.64%


### 3.3 Interpretation of Portfolio Results

      The development portfolio has an expected-loss rate of 10.19%, compared with 9.64% for the validation portfolio. Average expected loss per loan is $1,472.20 and $1,455.17, respectively.

      The validation rate is 0.55 percentage points lower. This difference should be treated as a descriptive comparison rather than evidence of improved model performance. The PD predictions for the development sample are generated from the model fitted on that same sample, whereas validation predictions are generated on held-out data. Differences in portfolio composition may also contribute to the result.

## 4. Limitations and Conclusion

### 4.1 Limitations

The analysis demonstrates how PD, LGD, and EAD can be combined to estimate expected credit loss. Several limitations should be considered when interpreting the results.

- **LGD assumptions:** LGD values are assigned by credit grade using an illustrative assumption framework. They are not estimated from loan-level realized losses.
- **Recovery-based proxy:** The exploratory recovery ratio uses original funded amounts as the denominator. It does not establish realized LGD because outstanding exposure at default is unavailable.
- **EAD proxy:** Original funded amounts are used instead of actual outstanding balances at default. Principal repayments and other balance adjustments are not captured.
- **PD model uncertainty:** Expected-loss estimates inherit the limitations of the PD model developed in Notebook 02.
- **Portfolio comparison:** Differences between development and validation expected-loss rates may reflect differences in portfolio composition as well as predicted default risk.
- **Regulatory applicability:** The calculations are illustrative and do not constitute a complete regulatory or accounting expected-credit-loss framework.

### 4.2 Conclusion

This notebook completes the initial credit-loss framework by establishing LGD assumptions and an EAD proxy and combining them with PD predictions.

The analysis demonstrates the relationship:

$$
EL_i = PD_i \times LGD_i \times EAD_i
$$

Using these inputs, the development portfolio produces an expected-loss rate of 10.19%, while the validation portfolio produces 9.64%.

These results provide an interpretable demonstration of expected credit loss estimation. Further refinement would require more reliable outstanding exposure data, realized loss and recovery information, and validated LGD estimates.